## GINN `best.pt` 推理与剖面对比

这个 notebook 会完成三件事：

1. 从 `experiments/ginn/checkpoints/best.pt` 加载训练好的模型。
2. 使用 `Trainer.predict_volume()` 对整个体做预测，得到阻抗体。
3. 在同一位置切出预测阻抗剖面和低频模型剖面，并排对比，同时给出差值剖面。


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from ginn.config import GINNConfig
from ginn.trainer import Trainer

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = False


In [ ]:
checkpoint_path = repo_root / "experiments" / "ginn" / "checkpoints" / "best.pt"
output_dir = repo_root / "data" / "_debug" / "ginn_prediction_compare@20260414"
output_dir.mkdir(parents=True, exist_ok=True)

slice_mode = "inline"  # 可选: "inline" 或 "xline"
slice_index = None  # None 表示默认取中间剖面
clip_percentiles = (1.0, 99.0)
ai_display_min = 0.0
ai_display_max = 20000.0

if not checkpoint_path.exists():
    raise FileNotFoundError(checkpoint_path)

checkpoint_path

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
cfg_payload = checkpoint["config"]
cfg = GINNConfig.from_dict(cfg_payload) if isinstance(cfg_payload, dict) else cfg_payload
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

trainer = Trainer(cfg)
trainer.model.load_state_dict(checkpoint["model_state_dict"])
trainer.model.eval()

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Epoch: {checkpoint['epoch']}")
print(f"Best loss: {checkpoint['best_loss']:.6f}")
print(f"Device: {trainer.device}")
print(f"Geometry: {trainer.geometry}")


In [ ]:
pred_volume = trainer.predict_volume()

n_il = int(trainer.geometry["n_il"])
n_xl = int(trainer.geometry["n_xl"])
n_sample = int(trainer.geometry["n_sample"])

lmf_volume = trainer.dataset._lmf_flat.reshape(n_il, n_xl, n_sample)
mask_volume = trainer.dataset._mask_flat.reshape(n_il, n_xl, n_sample)

prediction_path = output_dir / "pred_volume_best.npy"
np.save(prediction_path, pred_volume.astype(np.float32))

print("Prediction volume shape:", pred_volume.shape)
print("LMF volume shape:", lmf_volume.shape)
print("Mask volume shape:", mask_volume.shape)
print("Saved prediction to:", prediction_path)


In [ ]:
def resolve_slice_index(mode: str, index: int | None, geometry: dict) -> int:
    if mode not in {"inline", "xline"}:
        raise ValueError(f"slice_mode must be 'inline' or 'xline', got {mode!r}")
    size = int(geometry["n_il"] if mode == "inline" else geometry["n_xl"])
    if index is None:
        return size // 2
    if not (0 <= index < size):
        raise IndexError(f"slice_index={index} out of range for {mode} size={size}")
    return index


def extract_section(volume: np.ndarray, mode: str, index: int) -> np.ndarray:
    if mode == "inline":
        return volume[index, :, :].T
    if mode == "xline":
        return volume[:, index, :].T
    raise ValueError(mode)


def extract_mask_section(mask: np.ndarray, mode: str, index: int) -> np.ndarray:
    if mode == "inline":
        return mask[index, :, :].T
    if mode == "xline":
        return mask[:, index, :].T
    raise ValueError(mode)


def robust_limits(*arrays: np.ndarray, percentiles=(1.0, 99.0)) -> tuple[float, float]:
    values = np.concatenate([np.asarray(arr, dtype=np.float32).ravel() for arr in arrays])
    return tuple(np.percentile(values, percentiles))


resolved_slice_index = resolve_slice_index(slice_mode, slice_index, trainer.geometry)
pred_section = extract_section(pred_volume, slice_mode, resolved_slice_index)
lmf_section = extract_section(lmf_volume, slice_mode, resolved_slice_index)
mask_section = extract_mask_section(mask_volume, slice_mode, resolved_slice_index)
diff_section = pred_section - lmf_section

auto_vmin, auto_vmax = robust_limits(pred_section, lmf_section, percentiles=clip_percentiles)
shared_vmin = auto_vmin if ai_display_min is None else float(ai_display_min)
shared_vmax = auto_vmax if ai_display_max is None else float(ai_display_max)
diff_abs = np.percentile(np.abs(diff_section), clip_percentiles[1])

print(f"Slice mode: {slice_mode}")
print(f"Slice index: {resolved_slice_index}")
print(f"Shared display range: [{shared_vmin:.2f}, {shared_vmax:.2f}]")
print(f"Diff display abs range: +/-{diff_abs:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7), constrained_layout=True)

im0 = axes[0].imshow(pred_section, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax)
axes[0].set_title(f"Predicted AI | {slice_mode}={resolved_slice_index}")
axes[0].set_xlabel("Trace index")
axes[0].set_ylabel("Sample index")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(lmf_section, cmap="viridis", aspect="auto", origin="upper", vmin=shared_vmin, vmax=shared_vmax)
axes[1].set_title(f"LMF | {slice_mode}={resolved_slice_index}")
axes[1].set_xlabel("Trace index")
axes[1].set_ylabel("Sample index")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

im2 = axes[2].imshow(diff_section, cmap="seismic", aspect="auto", origin="upper", vmin=-diff_abs, vmax=diff_abs)
axes[2].set_title(f"Prediction - LMF | {slice_mode}={resolved_slice_index}")
axes[2].set_xlabel("Trace index")
axes[2].set_ylabel("Sample index")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

figure_path = output_dir / f"{slice_mode}_{resolved_slice_index:04d}_prediction_vs_lmf.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
print("Saved figure to:", figure_path)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
mask_image = np.where(mask_section, 1.0, 0.0)
im = ax.imshow(mask_image, cmap="gray_r", aspect="auto", origin="upper", vmin=0.0, vmax=1.0)
ax.set_title(f"Mask | {slice_mode}={resolved_slice_index}")
ax.set_xlabel("Trace index")
ax.set_ylabel("Sample index")
fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()
